> Public repository edition. The analysis uses the included **sample** by default. Full report results used the separate full dataset. Code paths and validation were cleaned for publication; original score formulas, pair split, model settings and evaluation definitions are retained. Outputs are cleared. See [reproducibility notes](../docs/reproducibility.md).

# Exploratory Data Analysis

We start by importing the packages needed for data handling and plotting. Then we load the processed base dataset that was created from the MinMax results. This dataset already contains the reconstructed raw min-max score, the sign-reversed score, the transformed paper score, the pair ID, and the label


In [ ]:
from pathlib import Path
import sys

# Start Jupyter in the repository root or its notebooks/ directory.
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "project_config.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks/ folder.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from project_config import get_paths

paths = get_paths()
print("Data mode:", paths.mode)
print("Input:", paths.input_path)
print("Local processed data:", paths.processed_dir)
print("Local evaluation outputs:", paths.evaluation_dir)

import pandas as pd
import numpy as np
from  matplotlib import pyplot as plt
from pathlib import Path

data_path = paths.processed_dir / "IdentityMF_MinMax_base_dataset.csv"

df = pd.read_csv(data_path)

df.head()

We first check the size of the dataset, the column types, and whether any values are missing. This helps confirm that the saved base dataset was loaded correctly

In [ ]:
print("Dataset shape:", df.shape)

print("\nColumn types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum())

Next, we check the label distribution. This tells us how many rows are same-compound comparisons and how many are different-compound comparisons. We expect the dataset to be highly imbalanced

In [ ]:
label_counts = df["label"].value_counts().sort_index()
label_props = df["label"].value_counts(normalize=True).sort_index()

print("Label counts:")
print(label_counts)

print("\nLabel proportions:")
print(label_props)

Now we check the compound-pair structure. Since each compound pair appears repeatedly because of the min-max iterations, this step helps confirm how many unique pairs exist and how many rows each pair contributes

In [ ]:
n_pairs = df["pair_id"].nunique()
rows_per_pair = df["pair_id"].value_counts()

print("Number of unique compound pairs:", n_pairs)

print("\nRows per pair summary:")
print(rows_per_pair.describe())

print("\nMost frequent pair counts:")
print(rows_per_pair.head())

We summarize the main score columns. This gives us the range and typical values of the reconstructed raw score, the sign-reversed score, and the transformed score from the paper

In [ ]:
score_columns = ["delta_raw", "r_overlap", "delta_prime"]

df[score_columns].describe()

Now we check how often the transformed paper score is equal to 1. This is important because the transformation clips all negative raw delta values to 1, which can remove useful information for calibration

In [ ]:
clipped_summary = (
    df.assign(is_clipped=df["delta_prime"] == 1)
      .groupby("label")["is_clipped"]
      .agg(["sum", "count", "mean"])
)

clipped_summary

This histogram shows the distribution of `r_overlap` for the two labels. Density normalization is used because the two classes have very different sizes.

In [ ]:
plt.figure(figsize=(8, 5))

for label in sorted(df["label"].unique()):
    subset = df[df["label"] == label]
    plt.hist(
        subset["r_overlap"],
        bins=50,
        alpha=0.5,
        density=True,
        label=f"label = {label}"
    )

plt.xlabel("r_overlap")
plt.ylabel("Density")
plt.title("Distribution of r_overlap by label")
plt.legend()
plt.show()

This histogram shows the original raw min-max score by label. Same-compound rows should generally appear more toward lower or negative raw delta values

In [ ]:
plt.figure(figsize=(8, 5))

for label in sorted(df["label"].unique()):
    subset = df[df["label"] == label]
    plt.hist(
        subset["delta_raw"],
        bins=50,
        alpha=0.5,
        density=True,
        label=f"label = {label}"
    )

plt.xlabel("delta_raw")
plt.ylabel("Density")
plt.title("Distribution of delta_raw by label")
plt.legend()
plt.show()

Finally, we use a boxplot to compare r_overlap between different-compound and same-compound rows. This gives a compact visual summary of whether the two classes are separated or strongly overlapping

In [ ]:
plt.figure(figsize=(7, 5))

data_to_plot = [
    df.loc[df["label"] == 0, "r_overlap"],
    df.loc[df["label"] == 1, "r_overlap"]
]

plt.boxplot(data_to_plot, tick_labels=["Different compound", "Same compound"])

plt.ylabel("r_overlap")
plt.title("r_overlap by label")
plt.show()

We now compare the main score columns separately for different-compound and same-compound rows. This helps us check whether the reconstructed scores behave differently for the two classes

In [ ]:
score_columns = ["delta_raw", "r_overlap", "delta_prime"]

df.groupby("label")[score_columns].describe()

This gives a simpler comparison of the average score values by label. For `r_overlap`, we expect the average value to be higher for same-compound rows

In [ ]:
df.groupby("label")[score_columns].mean()

We check how often `r_overlap` is positive for each label. Positive `r_overlap` means the between-sample similarity is stronger than the weakest within-sample similarity.

In [ ]:
positive_r_summary = (
    df.assign(r_positive=df["r_overlap"] > 0)
      .groupby("label")["r_positive"]
      .agg(["sum", "count", "mean"])
)

positive_r_summary

This check shows whether any same-compound rows have `delta_prime` less than 1. If this happens rarely, it confirms that the transformed score almost always collapses same-compound cases to exactly 1.

In [ ]:
same_compound_delta_prime = df.loc[df["label"] == 1, "delta_prime"]

print("Same-compound delta_prime summary:")
print(same_compound_delta_prime.describe())

print("\nNumber of same-compound rows with delta_prime < 1:")
print((same_compound_delta_prime < 1).sum())